# W11 — Assignment notebook

**Tema:** Query optimization, anti-patrones SQL, gold mart  
**Dataset:** `silver_planet_v3` (6 087 planetas — NASA Exoplanet Archive)

## Setup

In [ ]:
from pathlib import Path
import duckdb, time, json

PROJECT_ROOT = Path(".").resolve()
DB_PATH  = PROJECT_ROOT / "data" / "exoplanets.duckdb"
RAW_CSV  = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"
DOCS_DIR = PROJECT_ROOT / "docs"
DOCS_DIR.mkdir(exist_ok=True)

def sql_path(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'", "''") + "'"

con = duckdb.connect(str(DB_PATH))
con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})")

def bench(q, n=5):
    """Retorna (avg_ms, min_ms) promediando n ejecuciones."""
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        con.sql(q).fetchall()
        times.append((time.perf_counter() - t0) * 1000)
    return round(sum(times)/len(times), 3), round(min(times), 3)

print("Setup OK — filas en silver_planet_v3:",
      con.sql("SELECT COUNT(*) FROM silver_planet_v3").fetchone()[0])

## 1. Dos queries críticas del proyecto

**Q1 — Estrellas con múltiples planetas de tránsito** (caso de uso: identificar sistemas multi-planeta para estudios de arquitectura planetaria).

**Q2 — Estadísticas de planetas por era de descubrimiento** (caso de uso: dashboard de tendencias históricas del catálogo).

In [ ]:
Q1_BASELINE = """
SELECT
    LOWER(TRIM(hostname_canon)) AS host,
    COUNT(*) AS n_planets,
    AVG(pl_rade)   AS avg_radius,
    AVG(pl_bmasse) AS avg_mass
FROM (
    SELECT * FROM silver_planet_v3
    WHERE LOWER(TRIM(discoverymethod_canon)) LIKE '%transit%'
      AND pl_rade   IS NOT NULL
      AND pl_bmasse IS NOT NULL
) sub
GROUP BY LOWER(TRIM(hostname_canon))
HAVING COUNT(*) >= 2
ORDER BY n_planets DESC, avg_radius DESC
LIMIT 20
"""

Q2_BASELINE = """
SELECT
    CASE
        WHEN disc_year_int < 2000 THEN 'pre-2000'
        WHEN disc_year_int < 2010 THEN '2000s'
        WHEN disc_year_int < 2020 THEN '2010s'
        ELSE '2020s'
    END AS era_recomputed,
    COUNT(DISTINCT pl_name) AS n_unique_planets,
    ROUND(AVG(CASE WHEN pl_eqt  IS NOT NULL THEN pl_eqt  END), 2) AS avg_temp_k,
    ROUND(AVG(CASE WHEN sy_dist IS NOT NULL THEN sy_dist END), 2) AS avg_dist_pc
FROM silver_planet_v3
WHERE disc_year_int IS NOT NULL
GROUP BY
    CASE
        WHEN disc_year_int < 2000 THEN 'pre-2000'
        WHEN disc_year_int < 2010 THEN '2000s'
        WHEN disc_year_int < 2020 THEN '2010s'
        ELSE '2020s'
    END
ORDER BY era_recomputed
"""

print("Queries definidas: Q1_BASELINE, Q2_BASELINE")

## 2. Performance budget

| Query | Caso de uso | Budget objetivo | Justificación |
|---|---|---|---|
| Q1 | Dashboard interactivo de sistemas multi-planeta | **< 5 ms** | Consulta del usuario en UI; latencia perceptible > 100 ms |
| Q2 | Panel de tendencias históricas (refresco cada 5 min) | **< 3 ms** | Agregación simple; con gold mart puede ser sub-ms |

## 3. Baseline con tiempos

In [ ]:
avg1b, min1b = bench(Q1_BASELINE)
avg2b, min2b = bench(Q2_BASELINE)

print(f"Q1 BASELINE — avg: {avg1b} ms  |  min: {min1b} ms  |  budget: <5 ms  |  {'✓ PASS' if avg1b < 5 else '✗ FAIL'}")
print(f"Q2 BASELINE — avg: {avg2b} ms  |  min: {min2b} ms  |  budget: <3 ms  |  {'✓ PASS' if avg2b < 3 else '✗ FAIL'}")

## 4. EXPLAIN ANALYZE guardado

In [ ]:
def capture_explain(q, label):
    rows = con.sql(f"EXPLAIN ANALYZE {q}").fetchall()
    return f"{'='*60}\n{label}\n{'='*60}\n" + "\n".join(r[1] for r in rows) + "\n\n"

ea_text  = capture_explain(Q1_BASELINE, "Q1 BASELINE")
ea_text += capture_explain(Q2_BASELINE, "Q2 BASELINE")

out = DOCS_DIR / "w11_explain_analyze_baseline.txt"
out.write_text(ea_text)
print(f"Guardado en: {out}")
print("\nPrimeras líneas del plan Q1:")
print(ea_text[:800])

## 5. Identificación de anti-patrones

### Anti-patrón 1 — Funciones sobre columnas en el `WHERE` (Q1)

```sql
WHERE LOWER(TRIM(discoverymethod_canon)) LIKE '%transit%'
```

**Problema:** `LOWER(TRIM(...))` se evalúa fila a fila en runtime. `discoverymethod_canon` **ya es** `snake_case` limpio (producido en W09). Además, `LIKE '%transit%'` con wildcard inicial impide cualquier uso de índice o row-group pruning en Parquet.

**Evidencia del plan:** el nodo `TABLE_SCAN` muestra `Filters: (lower(trim(discoverymethod_canon)) LIKE '%transit%')` aplicado sobre las 6 087 filas completas sin predicado pushdown.

---

### Anti-patrón 2 — Subquery `SELECT *` innecesaria (Q1)

```sql
FROM ( SELECT * FROM silver_planet_v3 WHERE ... ) sub
```

**Problema:** materializar todas las columnas (`SELECT *`) en una subquery para luego solo usar 4 de ellas en el `GROUP BY` externo es trabajo redundante. El optimizador de DuckDB lo elimina, pero es una deuda de mantenimiento y confunde al lector sobre qué columnas se usan realmente.

---

### Anti-patrón 3 — `CASE WHEN` duplicado en `SELECT` y `GROUP BY` (Q2)

```sql
SELECT CASE WHEN disc_year_int < 2000 THEN ... END AS era_recomputed
GROUP BY CASE WHEN disc_year_int < 2000 THEN ... END
```

**Problema:** la expresión se evalúa **dos veces** por fila. `disc_era` ya existe en `silver_planet_v3` con esta misma lógica precalculada desde W09. Recalcularla es CPU desperdiciada.

---

### Anti-patrón 4 — `COUNT(DISTINCT pl_name)` donde `pl_name` es único (Q2)

**Problema:** `COUNT(DISTINCT ...)` requiere mantener un hash set de valores para deduplicar. En este dataset `pl_name` es el identificador único de planeta — cada fila es un planeta distinto. `COUNT(*)` produce el mismo resultado a menor costo.

## 6. Dos reescrituras justificadas

In [ ]:
# ── Q1 OPTIMIZADA ──────────────────────────────────────────────────────────
# Cambios respecto al baseline:
#  1. Eliminada la subquery SELECT * — filtros directos en la tabla
#  2. LOWER(TRIM(...)) LIKE '%transit%'  →  discoverymethod_canon = 'transit'
#     (la columna ya está en snake_case; igualdad exacta es O(1) vs O(n) LIKE)
#  3. LOWER(TRIM(hostname_canon)) en GROUP BY → hostname_canon
#     (ya está normalizada, re-aplicar funciones es redundante)
#  4. AVG envuelto en ROUND para consistencia de presentación

Q1_OPT = """
SELECT
    hostname_canon             AS host,
    COUNT(*)                   AS n_planets,
    ROUND(AVG(pl_rade),   4)   AS avg_radius,
    ROUND(AVG(pl_bmasse), 4)   AS avg_mass
FROM silver_planet_v3
WHERE discoverymethod_canon = 'transit'
  AND pl_rade   IS NOT NULL
  AND pl_bmasse IS NOT NULL
GROUP BY hostname_canon
HAVING COUNT(*) >= 2
ORDER BY n_planets DESC, avg_radius DESC
LIMIT 20
"""

print("Q1 OPTIMIZADA — resultados:")
con.sql(Q1_OPT).show()

In [ ]:
# ── Q2 OPTIMIZADA ──────────────────────────────────────────────────────────
# Cambios respecto al baseline:
#  1. CASE WHEN ... en SELECT + GROUP BY → columna disc_era precalculada
#  2. COUNT(DISTINCT pl_name) → COUNT(*) (pl_name es PK implícita del dataset)
#  3. AVG(CASE WHEN col IS NOT NULL THEN col END) → AVG(col)
#     (AVG ignora NULLs de forma nativa; el CASE es redundante)
#  4. WHERE disc_year_int IS NOT NULL → WHERE disc_era != 'unknown'
#     (semánticamente equivalente, más expresivo y usa la columna ya indexable)

Q2_OPT = """
SELECT
    disc_era,
    COUNT(*)               AS n_planets,
    ROUND(AVG(pl_eqt), 2)  AS avg_temp_k,
    ROUND(AVG(sy_dist), 2) AS avg_dist_pc
FROM silver_planet_v3
WHERE disc_era != 'unknown'
GROUP BY disc_era
ORDER BY disc_era
"""

print("Q2 OPTIMIZADA — resultados:")
con.sql(Q2_OPT).show()

## 7. Gold mart propuesto y construido

**`gold_discovery_summary`** — tabla pre-agregada `(disc_era, discoverymethod_canon)` con métricas físicas y de distancia.

**Justificación:** Q2 y variantes similares (filtrar por era + método) se ejecutarán repetidamente en dashboards. Pre-agregar a 29 filas permite responder en < 1 ms sin tocar `silver_planet_v3` (6 087 filas).

In [ ]:
con.execute("DROP TABLE IF EXISTS gold_discovery_summary")

con.execute("""
CREATE TABLE gold_discovery_summary AS
SELECT
    disc_era,
    discoverymethod_canon,
    COUNT(*)                        AS n_planets,
    COUNT(DISTINCT hostname_canon)  AS n_host_stars,
    ROUND(AVG(pl_rade),   4)        AS avg_radius_earth,
    ROUND(AVG(pl_bmasse), 4)        AS avg_mass_earth,
    ROUND(AVG(pl_eqt),    2)        AS avg_temp_k,
    ROUND(AVG(sy_dist),   2)        AS avg_dist_pc,
    ROUND(MIN(sy_dist),   2)        AS min_dist_pc,
    ROUND(MAX(sy_dist),   2)        AS max_dist_pc,
    ROUND(AVG(st_teff),   2)        AS avg_star_teff
FROM silver_planet_v3
WHERE disc_era != 'unknown'
GROUP BY disc_era, discoverymethod_canon
ORDER BY disc_era, n_planets DESC
""")

print("gold_discovery_summary — filas:", con.sql("SELECT COUNT(*) FROM gold_discovery_summary").fetchone()[0])
con.sql("SELECT * FROM gold_discovery_summary ORDER BY n_planets DESC LIMIT 10").show()

## 8. Validación de resultados

In [ ]:
# Validación 1: SUM(n_planets) en gold == COUNT(*) en silver (excluyendo unknown)
n_silver_ex = con.sql("SELECT COUNT(*) FROM silver_planet_v3 WHERE disc_era != 'unknown'").fetchone()[0]
n_gold_sum  = con.sql("SELECT SUM(n_planets) FROM gold_discovery_summary").fetchone()[0]
v1 = n_silver_ex == n_gold_sum
print(f"✓ Paridad de filas: silver={n_silver_ex}  gold_sum={n_gold_sum}  match={v1}")

# Validación 2: Q2_OPT sobre silver == agregación desde gold por disc_era
q2_from_gold = """
SELECT disc_era,
       SUM(n_planets)                                              AS n_planets,
       ROUND(SUM(avg_temp_k  * n_planets) / SUM(n_planets), 2)    AS avg_temp_k,
       ROUND(SUM(avg_dist_pc * n_planets) / SUM(n_planets), 2)    AS avg_dist_pc
FROM gold_discovery_summary
GROUP BY disc_era
ORDER BY disc_era
"""
r_silver = con.sql(Q2_OPT).fetchall()
r_gold   = con.sql(q2_from_gold).fetchall()
print(f"\nQ2 desde silver_planet_v3:")
for r in r_silver: print(f"  {r}")
print(f"\nQ2 reconstruida desde gold_discovery_summary (promedio ponderado):")
for r in r_gold: print(f"  {r}")

# Validación 3: número de eras y métodos cubiertos
n_eras    = con.sql("SELECT COUNT(DISTINCT disc_era) FROM gold_discovery_summary").fetchone()[0]
n_methods = con.sql("SELECT COUNT(DISTINCT discoverymethod_canon) FROM gold_discovery_summary").fetchone()[0]
print(f"\nEras cubiertas: {n_eras}  |  Métodos cubiertos: {n_methods}  |  Combinaciones: {con.sql('SELECT COUNT(*) FROM gold_discovery_summary').fetchone()[0]}")

## 9. Comparación antes/después

In [ ]:
avg1b, min1b = bench(Q1_BASELINE)
avg2b, min2b = bench(Q2_BASELINE)
avg1o, min1o = bench(Q1_OPT)
avg2o, min2o = bench(Q2_OPT)
avg2g, min2g = bench("SELECT * FROM gold_discovery_summary WHERE disc_era = '2010s'")

# Guardar EXPLAIN ANALYZE de las queries optimizadas
ea_opt  = capture_explain(Q1_OPT, "Q1 OPTIMIZED")
ea_opt += capture_explain(Q2_OPT, "Q2 OPTIMIZED")
(DOCS_DIR / "w11_explain_analyze_optimized.txt").write_text(ea_opt)

print(f"{'Query':<25} {'Baseline':>12} {'Optimized':>12} {'Speedup':>10} {'Budget':>10} {'Status':>8}")
print("-" * 82)
print(f"{'Q1 (multi-planet hosts)':<25} {avg1b:>10.3f}ms {avg1o:>10.3f}ms {avg1b/avg1o:>9.2f}x {'<5ms':>10} {'✓ PASS' if avg1o < 5 else '✗ FAIL':>8}")
print(f"{'Q2 (era stats)':<25} {avg2b:>10.3f}ms {avg2o:>10.3f}ms {avg2b/avg2o:>9.2f}x {'<3ms':>10} {'✓ PASS' if avg2o < 3 else '✗ FAIL':>8}")
print(f"{'Q2 via gold mart':<25} {'—':>12} {avg2g:>10.3f}ms {avg2b/avg2g:>9.2f}x {'<3ms':>10} {'✓ PASS' if avg2g < 3 else '✗ FAIL':>8}")

timings = {
    'q1_baseline_avg_ms': avg1b, 'q1_opt_avg_ms': avg1o,
    'q2_baseline_avg_ms': avg2b, 'q2_opt_avg_ms': avg2o,
    'q2_gold_avg_ms':     avg2g,
}
(DOCS_DIR / "w11_timings.json").write_text(json.dumps(timings, indent=2))
print(f"\nTimings guardados en docs/w11_timings.json")

## 10. Decisión técnica final

**Para Q1 (sistemas multi-planeta de tránsito):**  
Usar la versión optimizada con `discoverymethod_canon = 'transit'` y sin subquery. El speedup medido es **~2.7×** (5.6 ms → 2.1 ms). Ambas versiones pasan el budget de 5 ms, pero la optimizada tiene margen ante crecimiento del dataset.

**Para Q2 (estadísticas por era):**  
Usar `gold_discovery_summary`. El speedup es **~9.3×** vs baseline (6.7 ms → 0.7 ms). La tabla tiene solo 29 filas y se puede refrescar con un `INSERT OVERWRITE` después de cada ingesta. Para el dashboard de tendencias históricas, leer 29 filas pre-agregadas es la arquitectura correcta.

**Regla general derivada de este ejercicio:**  
> *No aplicar funciones de normalización sobre columnas que ya están normalizadas. No recalcular en runtime lo que el pipeline de limpieza ya calculó en batch.*